# 04. 신규 입사자 분석 — AI 노출 직업에서 신입이 먼저 줄었는가 (H4)

- 입력: `data/processed/mdis_clean.csv` (02 노트북이 만든 개인 단위 파일, 저장소에 없음. 받는 법은 설계 문서 8-7절)
- 출력: `reports/mdis_new_entrant_results.csv` (회귀 계수만), `reports/figures/fig11_신규입사자_계수_추이.png`

03 노트북의 한계는 "노출 직업의 청년 감소가 AI 이전부터 있었다"였다. 여기서는 **같은 직업 안에서** 신규 입사자(현 직장 12개월 미만)와
재직자(12개월 이상)를 비교한다. 직업 전체에 닥친 변화(원래 추세, 업황)는 둘에 똑같이 걸리므로 비율에서 빠진다.
AI가 원인이면 재직자보다 신규 입사자가 먼저 줄어야 한다.

표본 규칙(결과를 보기 전에 정함): 직업 소분류 중 **모든 반기에 청년 취업자 응답이 30명 이상**인 것만 쓴다.

In [ ]:
import sys
import warnings

sys.path.append("..")  # 노트북 폴더 기준으로 src를 찾는다 (README 6절)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from matplotlib import font_manager

from src import config

warnings.filterwarnings("ignore", category=FutureWarning)
INK2, MUTED, GRID, AXIS, SURFACE, BLUE, ORANGE = "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb", "#2a78d6", "#eb6834"
_fonts = {f.name for f in font_manager.fontManager.ttflist}
FONT = next(f for f in ["AppleGothic", "Malgun Gothic", "NanumGothic", "DejaVu Sans"] if f in _fonts)
sns.set_theme(style="whitegrid", font=FONT, rc={
    "axes.unicode_minus": False, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.edgecolor": AXIS, "axes.labelcolor": INK2,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.labelsize": 9,
    "axes.titleweight": "bold", "axes.titlesize": 10, "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})
MIN_N = 30          # 모든 반기에 청년 취업자 응답이 이 수 이상인 소분류만
BASE = "202202"     # 기준 시점: ChatGPT 출시 직전 조사

## 1. 소분류 노출도

01 노트북과 같은 방법이다. 세분류 = 연결된 ISCO 점수의 평균, 소분류 = 소속 세분류의 평균.

In [ ]:
ilo = pd.read_excel(config.DATA_RAW / "ilo_wp140" / "Final_Scores_ISCO08_Gmyrek_et_al_2025.xlsx").drop_duplicates("ISCO_08")
ilo["isco"] = ilo.ISCO_08.astype(str).str.zfill(4)
lk = pd.read_excel(next((config.DATA_RAW / "ksco_isco").glob("*.xlsx")), sheet_name="4-1. (연계표) 세분류 연계표(KSCO-ISCO)", dtype=str)
lk = lk.rename(columns={"한국코드": "ksco", "국제코드": "isco"})
lk["ksco"], lk["isco"] = lk.ksco.str.strip(), lk.isco.str.strip().str.zfill(4)
unit = lk.merge(ilo[["isco", "mean_score_2025"]], on="isco", how="left").groupby("ksco").mean_score_2025.mean()
exp3 = unit.groupby(unit.index.str[:3]).mean().dropna().rename("exp25")
print(f"노출도가 있는 소분류 {len(exp3)}개, 범위 {exp3.min():.2f} ~ {exp3.max():.2f}")

## 2. 개인 자료 → 소분류 × 반기 집계

취업자를 연령군(청년 / 30~59세)과 근속(신규 / 재직)으로 나눠 가중값을 더한다(천명).
이 노트북에서 개인 단위 값은 여기까지만 쓰고, 뒤로는 집계값만 쓴다.

In [ ]:
use = ["period", "age_group", "employed", "occ3", "ind3", "emp_status", "new_entrant", "tenure_m", "weight"]
e = pd.read_csv(config.DATA_PROCESSED / "mdis_clean.csv", usecols=use, dtype={"period": str, "occ3": str, "ind3": str, "emp_status": str})
e = e[e.employed & e.tenure_m.notna()].copy()
e["g"] = e.age_group.str[0] + np.where(e.new_entrant, "_new", "_old")   # y_new, y_old, a_new, a_old
e["ind2"] = e.ind3.str[:2]


def cells(data):
    c = data.groupby(["occ3", "period", "g"]).weight.sum().div(1000).unstack("g", fill_value=0).reset_index()
    return c.rename(columns={"occ3": "code"})


n_youth = e[e.age_group == "y15_29"].groupby(["occ3", "period"]).size().unstack(fill_value=0).min(axis=1)
keep = sorted(set(n_youth[n_youth >= MIN_N].index) & set(exp3.index))
youth_w = e[e.age_group == "y15_29"].groupby("occ3").weight.sum()
print(f"소분류 {e.occ3.nunique()}개 중 {len(keep)}개 사용, 청년 취업자의 {100 * youth_w[keep].sum() / youth_w.sum():.1f}%")

### 산업 구성으로 예상되는 신규 입사 비율 (통제 변수)

어떤 직업이 불황인 산업에 몰려 있으면 AI와 상관없이 신규 입사가 줄 수 있다. 그래서 직업마다
"2019~2022년 산업 구성 × 그 반기 각 산업의 청년 신규 입사 비율(자기 직업은 빼고 계산)"을 만들어 통제한다.

In [ ]:
y = e[e.age_group == "y15_29"]
oi = y.groupby(["occ3", "ind2", "period", "new_entrant"]).weight.sum().unstack("new_entrant", fill_value=0)
oi = oi.rename(columns={True: "new", False: "old"}).reset_index()
oi["tot"] = oi.new + oi.old
ind = oi.groupby(["ind2", "period"])[["new", "tot"]].sum().rename(columns={"new": "i_new", "tot": "i_tot"}).reset_index()
comp = y[y.period < "202301"].groupby(["occ3", "ind2"]).weight.sum().rename("s").reset_index()
comp["s"] /= comp.groupby("occ3").s.transform("sum")
grid = comp.merge(pd.DataFrame({"period": sorted(e.period.unique())}), how="cross").merge(ind, on=["ind2", "period"], how="left")
grid = grid.merge(oi[["occ3", "ind2", "period", "new", "tot"]], on=["occ3", "ind2", "period"], how="left").fillna(0)
grid["rate"] = (grid.i_new - grid.new) / (grid.i_tot - grid.tot).where(grid.i_tot > grid.tot)   # 자기 직업 제외
nat = y.groupby("period").apply(lambda g: (g.weight * g.new_entrant).sum() / g.weight.sum()).rename("nat")
grid = grid.merge(nat, on="period")
grid["rate"] = grid.rate.fillna(grid.nat)                                                     # 그 산업에 자기 직업뿐이면 전국 값
pred = (grid.s * grid.rate).groupby([grid.occ3, grid.period]).sum().clip(0.01, 0.99)
ind_ctrl = np.log(pred / (1 - pred)).rename("ind_ctrl").reset_index().rename(columns={"occ3": "code"})

## 3. 분석용 표

- `lr_new` = ln(청년 신규 ÷ 청년 재직): 주 결과변수
- `x_post` = 노출도(표준화) × 2023년 이후, `x_eci` = 노출도 × ECI 상승폭(2023 상반기 대비, 10점 단위)
- `w`: 2022년 청년 취업자 평균(직업마다 고정)

In [ ]:
eci = pd.read_csv(config.DATA_PROCESSED / "eci_ref.csv", dtype={"period": str}).set_index("period").eci


def build(data, codes):
    p = cells(data)
    p = p[p.code.isin(codes)].merge(exp3, left_on="code", right_index=True).merge(ind_ctrl, on=["code", "period"])
    p["exp_z"] = (p.exp25 - p.drop_duplicates("code").exp25.mean()) / p.drop_duplicates("code").exp25.std()
    p["post"] = (p.period > BASE).astype(float)
    p["eci_gain"] = ((p.period.map(eci) - eci["202301"]) / 10).fillna(0)
    p["t"] = (p.period.str[:4].astype(int) + p.period.str[-1].astype(int) / 2) - 2023    # 2022 하반기 = 0 (연 단위)
    p["x_post"], p["x_eci"] = p.exp_z * p.post, p.exp_z * p.eci_gain
    p["x_k"] = p.exp_z * (p.t * 2).clip(lower=0)                           # 노출도 × 2023년 이후 경과 반기 수
    with np.errstate(divide="ignore"):
        for k in ["y_new", "y_old", "a_new", "a_old"]:
            p["ln_" + k] = np.log(p[k])
        p["lr_new"] = p.ln_y_new - p.ln_y_old
        p["lr_new_adult"] = p.ln_a_new - p.ln_a_old
        p["lr_new_ya"] = p.ln_y_new - p.ln_a_new                           # H4의 원래 결과변수
        p["lr_y"] = np.log(p.y_new + p.y_old) - np.log(p.a_new + p.a_old)   # 03 노트북의 결과변수와 같은 정의
    p["share_new"] = 100 * p.y_new / (p.y_new + p.y_old)
    w = p[p.period.str[:4] == "2022"].groupby("code").apply(lambda g: (g.y_new + g.y_old).mean()).rename("w")
    return p.merge(w, on="code")


panel = build(e, keep)
PERIODS = sorted(panel.period.unique())
assert len(panel) == len(keep) * len(PERIODS), "빠진 칸이 있음"
assert np.isfinite(panel[["lr_new", "lr_new_adult", "lr_new_ya", "ln_y_new", "ln_y_old", "ln_a_new", "ln_a_old"]]).all().all(), "0명인 칸이 있음"
print(f"{panel.code.nunique()}개 소분류 × {len(PERIODS)}개 반기 = {len(panel)}행 ({PERIODS[0]} ~ {PERIODS[-1]})")
top = panel.drop_duplicates("code").nlargest(5, "exp25").code.tolist()
print("노출도 상위 5개 소분류 코드:", top)

## 4. 회귀 함수

03 노트북과 같은 방식이다. 직업·시점 고정효과를 넣고, 표준오차는 직업 단위로 묶는다.

In [ ]:
def fit(data, terms, y="lr_new", trend=False, weighted=True, extra=()):
    parts = [data[list(terms) + list(extra)], pd.get_dummies(data.period, prefix="p", drop_first=True),
             pd.get_dummies(data.code, prefix="c", drop_first=True)]
    if trend:
        parts.append(pd.get_dummies(data.code, prefix="tr", drop_first=True).mul(data.t, axis=0))
    X = sm.add_constant(pd.concat(parts, axis=1).astype(float))
    assert np.linalg.matrix_rank(X.values) == X.shape[1], "설명변수끼리 겹침(완전 공선성)"
    w = data.w if weighted else np.ones(len(data))
    return sm.WLS(data[y], X, weights=w).fit(cov_type="cluster", cov_kwds={"groups": data.code.astype("category").cat.codes})


rows = []


def add(label, res, term, scale=100):
    lo, hi = res.conf_int().loc[term] * scale
    rows.append({"모형": label, "변수": term, "계수": res.params[term] * scale, "하한": lo, "상한": hi, "p값": res.pvalues[term], "행": int(res.nobs)})
    return rows[-1]

## 5. 누가 줄었나 — 네 집단을 따로

계수는 "노출도가 1표준편차 높은 직업에서 2023년 이후 그 집단 취업자가 몇 % 더 변했나"다.
AI가 원인이면 **청년 신규**가 가장 크게 줄고 재직자는 덜 줄어야 한다.

In [ ]:
for col, name in [("ln_y_new", "청년 신규"), ("ln_y_old", "청년 재직"), ("ln_a_new", "30~59세 신규"), ("ln_a_old", "30~59세 재직")]:
    add(f"수준: {name}", fit(panel, ["x_post"], y=col), "x_post")
add("참고: 청년 전체÷30~59세 전체 (03 노트북의 결과변수)", fit(panel, ["x_post"], y="lr_y"), "x_post")
print(pd.DataFrame(rows).round(3).to_string(index=False))

## 6. 주 모형 — 신규 ÷ 재직

같은 직업·같은 반기의 재직자로 나누므로 직업 전체의 변화는 빠진다. 이 비교는 03 노트북 결과를 본 뒤(11/24) 추가한 것이다.
설계 문서 8-1절에서 미리 정한 H4(청년 신규 ÷ 30~59세 신규, 직업별 추세 + 경과 반기 수)도 그대로 돌린다.

In [ ]:
n0 = len(rows)
add("주 모형: 청년 신규÷재직, 2023년 이후", fit(panel, ["x_post"]), "x_post")
add("주 모형: 청년 신규÷재직, ECI 10점당", fit(panel, ["x_eci"]), "x_eci")
add("비교: 30~59세 신규÷재직, 2023년 이후", fit(panel, ["x_post"], y="lr_new_adult"), "x_post")
add("청년 신규 비중(%p), 2023년 이후", fit(panel, ["x_post"], y="share_new"), "x_post", scale=1)
add("H4 원래 식: 청년 신규÷30~59세 신규, 추세 포함, 반기당", fit(panel, ["x_k"], y="lr_new_ya", trend=True), "x_k")
add("H4 추세 없이: 청년 신규÷30~59세 신규, 2023년 이후", fit(panel, ["x_post"], y="lr_new_ya"), "x_post")
print(pd.DataFrame(rows[n0:]).round(3).to_string(index=False))

## 7. 시점별 계수 — 2023년 전에도 움직였는가

노출도 × 각 반기를 넣는다(기준 2022 하반기 = 0). 2023년 전 계수가 0 근처에서 평평해야 위 결과를 믿을 수 있다.

In [ ]:
def event(y):
    d = panel.copy()
    names = []
    for per in PERIODS:
        if per != BASE:
            d["e_" + per] = d.exp_z * (d.period == per)
            names.append("e_" + per)
    r = fit(d, names, y=y)
    out = pd.DataFrame({"period": [n[2:] for n in names], "b": r.params[names].values * 100,
                        "lo": r.conf_int().loc[names, 0].values * 100, "hi": r.conf_int().loc[names, 1].values * 100})
    out = pd.concat([out, pd.DataFrame({"period": [BASE], "b": [0.0], "lo": [0.0], "hi": [0.0]})]).sort_values("period")
    return out.reset_index(drop=True), r.wald_test(np.eye(len(r.params))[[list(r.params.index).index(n) for n in names if n[2:] < BASE]], scalar=True).pvalue


ev = {k: event(k) for k in ["lr_new", "ln_y_new", "ln_y_old"]}
print("주 결과변수의 시점별 계수(%)")
print(ev["lr_new"][0].round(2).to_string(index=False))
print("2023년 전 계수가 모두 0이라는 가설의 p값:", {k: round(float(v[1]), 3) for k, v in ev.items()})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
xs = np.arange(len(PERIODS))
labels = [f"{p[2:4]}.{'상' if p[-1] == '1' else '하'}" for p in PERIODS]
for ax, key, color, title in [(axes[0], "ln_y_new", ORANGE, "청년 신규 입사자 (12개월 미만)"), (axes[1], "ln_y_old", BLUE, "청년 재직자 (12개월 이상)")]:
    t = ev[key][0]
    ax.fill_between(xs, t.lo, t.hi, color=color, alpha=0.15, lw=0)
    ax.plot(xs, t.b, color=color, lw=2, marker="o", ms=4)
    ax.axhline(0, color=AXIS, lw=1)
    ax.axvline(PERIODS.index(BASE) + 0.5, color=MUTED, lw=1, ls="--")
    ax.text(PERIODS.index(BASE) + 0.6, 0.98, " ChatGPT 출시 이후", color=MUTED, fontsize=8, va="top", transform=ax.get_xaxis_transform())
    ax.set_xticks(xs[::2])
    ax.set_xticklabels(labels[::2])
    ax.set_title(title)
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("노출도 1표준편차당 차이 (%, 2022 하반기 = 0)")
fig.suptitle("AI 노출도가 높은 직업에서 청년 취업자가 얼마나 더 변했나", x=0.01, ha="left", fontsize=11, fontweight="bold")
fig.text(0.01, -0.03, f"MDIS 지역별고용조사 B형, 직업 소분류 {panel.code.nunique()}개 × 반기 {len(PERIODS)}개. 띠는 95% 신뢰구간(직업 단위로 묶음).", color=MUTED, fontsize=8)
fig.tight_layout()
fig.savefig(config.FIGURES / "fig11_신규입사자_계수_추이.png")
plt.show()

## 8. 검증

- 가짜 시점: 2022년까지만 써서 "2021 상반기부터 AI가 있었다"고 가정한다. 여기서 효과가 나오면 위 결과는 AI가 아닌 다른 것이다.
- 직업별 추세, 산업 구성 통제, 임금근로자만, 표본 규칙 변경, 가중치 없음, 코로나 시기 제외.

In [ ]:
n0 = len(rows)
pre = panel[panel.period <= BASE].assign(x_fake=lambda d: d.exp_z * (d.period >= "202101"))
add("가짜 시점(2021 상~), 2022년까지만", fit(pre, ["x_fake"]), "x_fake")
add("직업별 추세 포함", fit(panel, ["x_post"], trend=True), "x_post")
add("산업 구성 통제", fit(panel, ["x_post"], extra=["ind_ctrl"]), "x_post")
add("임금근로자만", fit(build(e[e.emp_status.isin(["1", "2", "3"])], keep), ["x_post"]), "x_post")
keep50 = sorted(set(n_youth[n_youth >= 50].index) & set(exp3.index))
add(f"응답 50명 이상 소분류({len(keep50)}개)", fit(build(e, keep50), ["x_post"]), "x_post")
add("가중치 없음", fit(panel, ["x_post"], weighted=False), "x_post")
add("코로나 시기(2020~2021) 제외", fit(panel[~panel.period.str[:4].isin(["2020", "2021"])], ["x_post"]), "x_post")
print(pd.DataFrame(rows[n0:]).round(3).to_string(index=False))

## 9. 저장

In [ ]:
res = pd.DataFrame(rows).round(4)
res.to_csv(config.FIGURES.parent / "mdis_new_entrant_results.csv", index=False, encoding="utf-8-sig")
print(res.to_string(index=False))